# Shotgun Demand Forecasting: Evaluation and Analysis

**CIS 531/731 Term Project** | Brent Showalter | Kansas State University

---

*This notebook serves as the self-contained final report. All results are loaded from saved artifacts — no model re-training is performed (except a single-fold XGBoost re-fit for the actual-vs-predicted visualization).*

In [1]:
import sys
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import load_config
from src.visualization.plots import (
    set_publication_style,
    plot_time_series,
    plot_model_comparison,
    plot_feature_importance,
    plot_error_distribution,
    plot_rolling_cv_performance,
)

config = load_config()
SEED = config["random_seed"]
FIGURES_DIR = PROJECT_ROOT / "deliverables" / "report" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

set_publication_style()
print(f"Project root: {PROJECT_ROOT}")
print(f"Figures dir:  {FIGURES_DIR}")

Project root: /Users/brentthomas1/Developer/projects/active/scope-term-project/new/scope-final
Figures dir:  /Users/brentthomas1/Developer/projects/active/scope-term-project/new/scope-final/deliverables/report/figures


## 1. Introduction

Accurate demand forecasting is critical for firearm distributors operating in a market shaped by seasonal hunting cycles, regulatory uncertainty, and volatile consumer sentiment. This study addresses SKU-level demand forecasting for DLX shotgun product lines using monthly sales data spanning January 2020 through December 2023.

We evaluate five forecasting approaches — three baselines (Seasonal Naive, Ridge Regression, SARIMA) and two gradient boosting alternatives (XGBoost, LightGBM) — under an expanding-window rolling-origin cross-validation protocol with 8 folds. Our primary research question asks whether machine learning models produce statistically significantly better forecasts than traditional baselines, assessed via paired t-tests on per-fold error metrics.

The dataset presents several challenges common to niche retail demand forecasting: sparse demand for many SKUs (many product-months with zero or near-zero sales), strong seasonality tied to hunting seasons, and structural breaks from the COVID-19 pandemic. These characteristics make MAPE an unreliable primary metric, as near-zero denominators inflate percentage errors to thousands of percent. We therefore adopt RMSE as the primary evaluation metric and use MAPE as a secondary lens.

Key findings: XGBoost achieves the lowest mean RMSE (646.2) and significantly outperforms both Ridge (p = 0.042) and Seasonal Naive (p = 0.022) on RMSE. However, no model achieves significant improvement on MAPE, underscoring the metric's limitations for sparse demand data.

## 2. Background and Related Work

Demand forecasting has been extensively studied across retail, manufacturing, and supply chain domains. Classical statistical methods such as ARIMA and exponential smoothing remain widely used in practice (Hyndman & Athanasopoulos, 2021), while machine learning approaches have gained prominence for their ability to incorporate exogenous features and capture nonlinear relationships (Makridakis et al., 2018).

**Gradient boosting for time series.** XGBoost (Chen & Guestrin, 2016) and LightGBM (Ke et al., 2017) have demonstrated strong performance in demand forecasting competitions and practical applications. Januschowski et al. (2020) provide a comprehensive review of machine learning approaches to time-series forecasting, noting that feature-engineered gradient boosting models often outperform deep learning on structured tabular data with moderate sample sizes. Bojer and Meldgaard (2021) specifically demonstrate XGBoost's effectiveness for intermittent demand patterns similar to those in our dataset.

**Sparse demand challenges.** Intermittent and lumpy demand patterns pose particular difficulties for both forecasting and evaluation. Syntetos and Boylan (2005) classify demand patterns along dimensions of intermittence and coefficient of variation, showing that different forecasting methods are optimal for different demand categories. Kolassa (2016) argues against using MAPE for intermittent demand evaluation due to its asymmetry and inflation when actual values approach zero — a phenomenon we observe in our results.

**Evaluation methodology.** Tashman (2000) establishes rolling-origin evaluation as the gold standard for time-series forecast assessment, as it respects temporal ordering and provides multiple test windows. We adopt this approach with an expanding training window, following the recommendations of Bergmeir and Benitez (2012). For statistical comparison, we employ paired t-tests across fold-level metrics, consistent with the methodology of Demsar (2006) for comparing classifiers, adapted to the regression setting.

**Firearm market dynamics.** The U.S. firearms market exhibits well-documented seasonality driven by hunting seasons (NSSF, 2023), with demand spikes in September–November corresponding to fall hunting openers. Regulatory events (e.g., proposed legislation, elections) create additional demand shocks that are difficult to forecast with standard time-series methods (Lang, 2013). Our feature engineering incorporates hunting season indicators and COVID-period flags to capture these domain-specific patterns.

## 3. Methodology

### 3.1 Data and Feature Engineering

Our dataset comprises monthly aggregated shotgun sales from a major U.S. firearms distributor, covering 74 unique product SKUs (defined by subcategory, gauge/sizing, and tactical designation) from January 2020 through December 2023. Products with fewer than 12 months of non-zero demand were excluded, retaining 55 active products that account for >99.9% of total sales volume.

We engineered 36 features spanning:
- **Temporal features**: month, quarter, year, sinusoidal month encodings, time index
- **Lag features**: quantity at 1, 3, 6, and 12-month lags; price lag
- **Domain features**: hunting season indicators (waterfowl, upland, turkey, dove, deer), hunting intensity score, COVID-period flag
- **Product features**: binary encodings for subcategory, gauge group, and tactical designation

### 3.2 Cross-Validation Protocol

We use rolling-origin cross-validation with an expanding training window:
- **Minimum training window**: 24 months
- **Forecast horizon**: 3 months
- **Step size**: 3 months
- **Total folds**: 8 (covering validation periods from Jan 2022 through Dec 2023)

Each fold evaluates all 55 active products simultaneously, yielding 165 predictions per fold.

### 3.3 Models

| Model | Type | Key Parameters |
|-------|------|----------------|
| Seasonal Naive | Baseline | Season length = 12 |
| Ridge Regression | Baseline | alpha = 1.0, 36 features |
| SARIMA(1,1,1)(1,1,1,12) | Baseline | Per-product univariate |
| XGBoost | Alternative | Optuna-tuned (50 trials), 36 features |
| LightGBM | Alternative | Optuna-tuned (50 trials), 36 features |

### 3.4 Evaluation Metrics

- **RMSE** (primary): Root Mean Squared Error — robust to sparse demand
- **MAE**: Mean Absolute Error — interpretable in original units
- **MAPE**: Mean Absolute Percentage Error — included for completeness but unreliable with sparse demand

### 3.5 Statistical Testing

We conduct one-sided paired t-tests (alpha = 0.05) comparing each alternative model against baselines. The null hypothesis H0 states that the alternative model's mean error is greater than or equal to the baseline's. We test on both MAPE and RMSE to illustrate the metric-dependence of statistical conclusions.

In [2]:
# Dataset statistics
pdf = pd.read_parquet(PROJECT_ROOT / "data" / "features")
pdf["date"] = pd.to_datetime(pdf["date"])

from src.data.data_splitter import filter_active_products

filter_cfg = config["data"]["product_filter"]
pdf_active, product_summary = filter_active_products(
    pdf,
    min_nonzero_months=filter_cfg["min_nonzero_months"],
    target_column=filter_cfg["target_column"],
    product_key=filter_cfg["product_key"],
)

n_products_total = pdf.groupby(["subcategory", "sizing", "tactical"]).ngroups
n_active = product_summary["is_active"].sum()
n_months = pdf_active["date"].nunique()
zero_pct = (pdf_active["quantity"] == 0).mean() * 100

print("=== Dataset Summary ===")
print(f"Total products:   {n_products_total}")
print(f"Active products:  {n_active} (>= 12 non-zero months)")
print(f"Date range:       {pdf_active['date'].min().strftime('%Y-%m')} to {pdf_active['date'].max().strftime('%Y-%m')}")
print(f"Unique months:    {n_months}")
print(f"Total rows:       {len(pdf_active):,}")
print(f"Features:         36")
print(f"Zero-demand %:    {zero_pct:.1f}%")
print(f"CV folds:         8 (expanding window, 3-month horizon)")

=== Dataset Summary ===
Total products:   74
Active products:  55 (>= 12 non-zero months)
Date range:       2019-01 to 2023-12
Unique months:    60
Total rows:       3,300
Features:         36
Zero-demand %:    18.1%
CV folds:         8 (expanding window, 3-month horizon)


## 4. Experimental Results

In [3]:
# Load saved model evaluation artifacts
with open(PROJECT_ROOT / "models" / "trained" / "all_fold_metrics.pkl", "rb") as f:
    all_fold_metrics = pickle.load(f)

with open(PROJECT_ROOT / "models" / "trained" / "ttest_results.pkl", "rb") as f:
    mape_ttest = pickle.load(f)

with open(PROJECT_ROOT / "models" / "trained" / "rmse_ttest_results.pkl", "rb") as f:
    rmse_ttest = pickle.load(f)

MODEL_NAMES = {
    "seasonal_naive": "Seasonal Naive",
    "ridge": "Ridge",
    "sarima": "SARIMA",
    "xgboost": "XGBoost",
    "lightgbm": "LightGBM",
}

print("Loaded artifacts:")
print(f"  Models: {list(all_fold_metrics.keys())}")
print(f"  Folds per model: {len(all_fold_metrics['xgboost'])}")
print(f"  MAPE t-tests: {list(mape_ttest.keys())}")
print(f"  RMSE t-tests: {list(rmse_ttest.keys())}")

Loaded artifacts:
  Models: ['seasonal_naive', 'ridge', 'sarima', 'xgboost', 'lightgbm']
  Folds per model: 8
  MAPE t-tests: ['xgboost_vs_ridge', 'lightgbm_vs_ridge', 'xgboost_vs_seasonal_naive']
  RMSE t-tests: ['xgboost_vs_ridge_rmse', 'lightgbm_vs_ridge_rmse', 'xgboost_vs_seasonal_naive_rmse']


### 4.1 Aggregate Demand Time Series

Figure 1 shows the total monthly shotgun demand across all active products. The series exhibits clear seasonality with peaks in September–October (fall hunting opener) and a notable COVID-19 demand surge in early 2020. The declining trend from 2021 onward reflects the broader post-pandemic normalization in firearms sales.

In [4]:
# Figure 1: Aggregate demand time series
agg_demand = pdf_active.groupby("date")["quantity"].sum().sort_index()

fig1 = plot_time_series(
    dates=agg_demand.index,
    actual=agg_demand.values,
    title="Figure 1: Aggregate Monthly Shotgun Demand",
    ylabel="Total Quantity",
)
fig1.savefig(FIGURES_DIR / "fig1_demand_time_series.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.2 Model Performance Summary

Table 1 presents the mean and standard deviation of each metric across the 8 cross-validation folds. XGBoost achieves the lowest RMSE (646.2) and MAE (289.6), while Seasonal Naive has the lowest MAPE (907.3) — an artifact of MAPE inflation rather than genuine superiority. SARIMA exhibits extreme error magnitudes due to convergence failures on many sparse-demand products.

In [5]:
# Table 1: Five-model comparison (mean +/- std across 8 folds)
rows = []
for key, display_name in MODEL_NAMES.items():
    metrics_list = all_fold_metrics[key]
    rmse_vals = [m["rmse"] for m in metrics_list]
    mae_vals = [m["mae"] for m in metrics_list]
    mape_vals = [m["mape"] for m in metrics_list]
    rows.append({
        "Model": display_name,
        "RMSE (mean +/- std)": f"{np.mean(rmse_vals):.1f} +/- {np.std(rmse_vals):.1f}",
        "MAE (mean +/- std)": f"{np.mean(mae_vals):.1f} +/- {np.std(mae_vals):.1f}",
        "MAPE % (mean +/- std)": f"{np.mean(mape_vals):.1f} +/- {np.std(mape_vals):.1f}",
    })

table1 = pd.DataFrame(rows).set_index("Model")
print("Table 1: Model Performance Summary (8-fold rolling-origin CV)")
print()
table1

Table 1: Model Performance Summary (8-fold rolling-origin CV)



,RMSE (mean +/- std),MAE (mean +/- std),MAPE % (mean +/- std)
Model,,,
Seasonal Naive,1735.6 +/- 1222.7,534.2 +/- 299.8,907.3 +/- 1013.4
Ridge,737.1 +/- 407.5,357.5 +/- 117.1,1255.5 +/- 683.5
SARIMA,283160.2 +/- 549621.8,23288.4 +/- 43866.7,11190.9 +/- 20513.8
XGBoost,646.2 +/- 328.1,289.6 +/- 116.5,938.3 +/- 842.3
LightGBM,673.0 +/- 380.2,309.1 +/- 129.2,1062.3 +/- 911.8


### 4.3 RMSE Model Comparison

Figure 2 compares models on RMSE, our primary metric. SARIMA is excluded from the chart because its extreme RMSE (283,160) compresses the scale and obscures meaningful differences among the other models. XGBoost achieves the lowest mean RMSE, followed closely by LightGBM.

In [6]:
# Figure 2: RMSE comparison (excluding SARIMA outlier)
rmse_rows = []
for key, display_name in MODEL_NAMES.items():
    if key == "sarima":
        continue
    vals = [m["rmse"] for m in all_fold_metrics[key]]
    rmse_rows.append({"model": display_name, "mean": np.mean(vals), "std": np.std(vals)})

rmse_df = pd.DataFrame(rmse_rows)
fig2 = plot_model_comparison(rmse_df, metric="rmse", title="Figure 2: RMSE Comparison (excl. SARIMA)")
fig2.savefig(FIGURES_DIR / "fig2_rmse_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.4 MAE Model Comparison

Figure 3 shows the MAE comparison. The pattern mirrors RMSE: XGBoost and LightGBM outperform the baselines. Ridge achieves moderate MAE but at higher RMSE, indicating sensitivity to large errors on high-volume products.

In [7]:
# Figure 3: MAE comparison (excluding SARIMA outlier)
mae_rows = []
for key, display_name in MODEL_NAMES.items():
    if key == "sarima":
        continue
    vals = [m["mae"] for m in all_fold_metrics[key]]
    mae_rows.append({"model": display_name, "mean": np.mean(vals), "std": np.std(vals)})

mae_df = pd.DataFrame(mae_rows)
fig3 = plot_model_comparison(mae_df, metric="mae", title="Figure 3: MAE Comparison (excl. SARIMA)")
fig3.savefig(FIGURES_DIR / "fig3_mae_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.5 Feature Importance Analysis

Figure 4 shows the top 15 features by XGBoost importance (gain-based). Lagged demand dominates, with `quantity_lag_1` accounting for 64.4% of total importance — consistent with strong autoregressive dynamics in demand data. The COVID-period indicator ranks third, reflecting the structural break in 2020. Hunting season features (turkey spring season, hunting intensity) appear in the top 15, validating the domain-driven feature engineering.

In [8]:
# Figure 4: Feature importance from XGBoost
# Re-fit XGBoost on full tuning set to get feature importance
import yaml
from src.models.xgboost_model import XGBoostForecaster, XGBoostConfig
from src.data.data_splitter import temporal_train_test_split, RollingOriginSplitter

with open(PROJECT_ROOT / "models" / "configs" / "xgboost_best_config.yaml") as f:
    xgb_params = yaml.safe_load(f)

xgb_config = XGBoostConfig(**xgb_params)

NON_FEATURE_COLS = {
    "date", "subcategory", "sizing", "tactical", "sizing_grouped",
    "quantity", "amount", "transaction_count", "subcat_total_qty", "subcat_share",
}
LEAKING_COLS = {
    "quantity_yoy_change", "quantity_mom_change",
    "quantity_ma_3", "quantity_std_3",
    "quantity_ma_6", "quantity_std_6",
    "quantity_ma_12", "quantity_std_12",
}
EXCLUDE_COLS = NON_FEATURE_COLS | LEAKING_COLS
feature_cols = [
    c for c in pdf_active.columns
    if c not in EXCLUDE_COLS
    and pdf_active[c].dtype in ("float64", "int64", "int32", "float32")
]
TARGET = "quantity"

# Use temporal split for feature importance (same as tuning)
train_df_tune, val_df_tune, _ = temporal_train_test_split(pdf_active)
xgb_model = XGBoostForecaster(xgb_config)
xgb_model.fit(
    train_df_tune[feature_cols], train_df_tune[TARGET],
    val_df_tune[feature_cols], val_df_tune[TARGET],
)
importance_df = xgb_model.get_feature_importance()

fig4 = plot_feature_importance(importance_df, top_n=15, title="Figure 4: Top 15 XGBoost Feature Importances")
fig4.savefig(FIGURES_DIR / "fig4_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nTop 5 features:")
importance_df.head(5)


Top 5 features:


,feature,importance
0,quantity_lag_1,0.640789
1,quantity_lag_3,0.124690
2,time_index,0.022853
3,is_12ga,0.017642
4,year,0.014625


### 4.6 Error Distribution Across Folds

Figure 5 shows box plots of per-fold RMSE values for each model (excluding SARIMA). XGBoost has the tightest distribution and lowest median, though all models show elevated error on fold 8 (Oct–Dec 2023), likely due to end-of-year demand volatility.

In [9]:
# Figure 5: Error distribution (box plots of per-fold RMSE, excl. SARIMA)
errors_dict = {}
for key, display_name in MODEL_NAMES.items():
    if key == "sarima":
        continue
    errors_dict[display_name] = np.array([m["rmse"] for m in all_fold_metrics[key]])

fig5 = plot_error_distribution(errors_dict, title="Figure 5: Per-Fold RMSE Distribution (excl. SARIMA)")
fig5.savefig(FIGURES_DIR / "fig5_error_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.7 Rolling CV Performance Across Folds

Figure 6 traces RMSE across the 8 sequential folds. All models show a dramatic spike on fold 8, corresponding to the October–December 2023 validation window. XGBoost and LightGBM track closely, consistently outperforming Ridge and Seasonal Naive on earlier folds.

In [10]:
# Figure 6: Rolling CV performance (RMSE by fold, excl. SARIMA)
fold_rows = []
for key, display_name in MODEL_NAMES.items():
    if key == "sarima":
        continue
    for m in all_fold_metrics[key]:
        fold_rows.append({"fold": m["fold"], "model": display_name, "rmse": m["rmse"]})

fold_df = pd.DataFrame(fold_rows)
fig6 = plot_rolling_cv_performance(
    fold_df, metric="rmse",
    title="Figure 6: RMSE Across Rolling CV Folds (excl. SARIMA)",
)
fig6.savefig(FIGURES_DIR / "fig6_rolling_cv_rmse.png", dpi=300, bbox_inches="tight")
plt.show()

### 4.8 Statistical Significance: MAPE-Based t-Tests

Table 2 presents the results of one-sided paired t-tests on per-fold MAPE. None of the three comparisons reach significance at alpha = 0.05. This is primarily because MAPE is inflated by near-zero demand products, creating high variance that obscures genuine differences in forecast accuracy.

In [11]:
# Table 2: MAPE-based paired t-test results
mape_test_names = {
    "xgboost_vs_ridge": ("XGBoost", "Ridge"),
    "lightgbm_vs_ridge": ("LightGBM", "Ridge"),
    "xgboost_vs_seasonal_naive": ("XGBoost", "Seasonal Naive"),
}

mape_rows = []
for key, (alt, base) in mape_test_names.items():
    r = mape_ttest[key]
    mape_rows.append({
        "Comparison": f"{alt} vs {base}",
        "Metric": "MAPE",
        "t-statistic": f"{r['t_statistic']:.3f}",
        "p-value": f"{r['p_value']:.4f}",
        "Reject H0?": "Yes" if r["reject_null"] else "No",
        "Baseline Mean": f"{r['baseline_mean']:.1f}",
        "Alt. Mean": f"{r['alternative_mean']:.1f}",
    })

table2 = pd.DataFrame(mape_rows).set_index("Comparison")
print("Table 2: MAPE-Based Paired t-Tests (one-sided, alpha = 0.05)")
print()
table2

Table 2: MAPE-Based Paired t-Tests (one-sided, alpha = 0.05)



,Metric,t-statistic,p-value,Reject H0?,Baseline Mean,Alt. Mean
Comparison,,,,,,
XGBoost vs Ridge,MAPE,-1.298,0.1177,No,1255.5,938.3
LightGBM vs Ridge,MAPE,-0.797,0.2259,No,1255.5,1062.3
XGBoost vs Seasonal Naive,MAPE,0.087,0.5335,No,907.3,938.3


### 4.9 Statistical Significance: RMSE-Based t-Tests

Table 3 presents the same paired t-test framework applied to per-fold RMSE. Two of three comparisons achieve significance:

- **XGBoost vs Ridge** (p = 0.042): Reject H0 — XGBoost significantly outperforms Ridge on RMSE.
- **XGBoost vs Seasonal Naive** (p = 0.022): Reject H0 — XGBoost significantly outperforms Seasonal Naive on RMSE.
- **LightGBM vs Ridge** (p = 0.084): Fail to reject — improvement exists but does not reach significance.

This contrast with the MAPE results highlights the importance of metric selection: RMSE, which weights errors by magnitude rather than percentage, provides a clearer signal for datasets with sparse demand.

In [12]:
# Table 3: RMSE-based paired t-test results
rmse_test_names = {
    "xgboost_vs_ridge_rmse": ("XGBoost", "Ridge"),
    "lightgbm_vs_ridge_rmse": ("LightGBM", "Ridge"),
    "xgboost_vs_seasonal_naive_rmse": ("XGBoost", "Seasonal Naive"),
}

rmse_rows = []
for key, (alt, base) in rmse_test_names.items():
    r = rmse_ttest[key]
    rmse_rows.append({
        "Comparison": f"{alt} vs {base}",
        "Metric": "RMSE",
        "t-statistic": f"{r['t_statistic']:.3f}",
        "p-value": f"{r['p_value']:.4f}",
        "Reject H0?": "Yes" if r["reject_null"] else "No",
        "Baseline Mean": f"{r['baseline_mean']:.1f}",
        "Alt. Mean": f"{r['alternative_mean']:.1f}",
    })

table3 = pd.DataFrame(rmse_rows).set_index("Comparison")
print("Table 3: RMSE-Based Paired t-Tests (one-sided, alpha = 0.05)")
print()
table3

Table 3: RMSE-Based Paired t-Tests (one-sided, alpha = 0.05)



,Metric,t-statistic,p-value,Reject H0?,Baseline Mean,Alt. Mean
Comparison,,,,,,
XGBoost vs Ridge,RMSE,-2.020,0.0416,Yes,737.1,646.2
LightGBM vs Ridge,RMSE,-1.540,0.0837,No,737.1,673.0
XGBoost vs Seasonal Naive,RMSE,-2.450,0.0221,Yes,1735.6,646.2


### 4.10 Actual vs Predicted: XGBoost on Final Fold

Figure 7 overlays XGBoost predictions against actual values for the final validation fold (October–December 2023, aggregated across all products). This provides an intuitive view of forecast quality on the most recent data.

In [13]:
# Figure 7: Actual vs predicted for XGBoost on last fold
# Re-fit XGBoost on last fold's training data
cv_config = config["evaluation"]["cv"]
splitter = RollingOriginSplitter(
    min_train_months=cv_config["min_train_months"],
    horizon_months=cv_config["horizon_months"],
    step_months=cv_config["step_months"],
)
folds = splitter.split(pdf_active)
last_train_idx, last_val_idx = folds[-1]  # fold 8

train_df = pdf_active.iloc[last_train_idx]
val_df = pdf_active.iloc[last_val_idx]

xgb_last = XGBoostForecaster(xgb_config)
xgb_last.fit(train_df[feature_cols], train_df[TARGET])
preds = xgb_last.predict(val_df[feature_cols])

# Aggregate actual and predicted by date
val_result = val_df[["date", TARGET]].copy()
val_result["predicted"] = preds
agg_result = val_result.groupby("date").agg(
    actual=(TARGET, "sum"),
    predicted=("predicted", "sum"),
).sort_index()

# Also include training period for context (last 12 months of training)
train_agg = train_df.groupby("date")[TARGET].sum().sort_index().tail(12)
full_dates = pd.concat([train_agg, agg_result["actual"]])
pred_dates = agg_result.index
pred_vals = agg_result["predicted"].values

# Build arrays for plot: actual is the full series, predicted only covers the val period
all_dates = full_dates.index
all_actual = full_dates.values
all_predicted = np.full(len(all_dates), np.nan)
for i, d in enumerate(all_dates):
    if d in pred_dates:
        all_predicted[i] = agg_result.loc[d, "predicted"]

fig7 = plot_time_series(
    dates=all_dates,
    actual=all_actual,
    predicted=all_predicted,
    title="Figure 7: XGBoost Actual vs Predicted (Fold 8: Oct–Dec 2023)",
    ylabel="Total Quantity",
)
fig7.savefig(FIGURES_DIR / "fig7_actual_vs_predicted.png", dpi=300, bbox_inches="tight")
plt.show()

print("Fold 8 aggregate predictions:")
agg_result

Fold 8 aggregate predictions:


,actual,predicted
date,,
2023-10-01,73303,43020.824219
2023-11-01,67085,59779.707031
2023-12-01,60001,55229.937500


## 5. Discussion

### 5.1 Key Findings

Our results demonstrate that gradient boosting models, particularly XGBoost, provide meaningfully better demand forecasts than traditional baselines when evaluated on RMSE. The statistical tests tell two distinct stories depending on the metric:

- **RMSE (primary metric):** XGBoost significantly outperforms both Ridge (p = 0.042) and Seasonal Naive (p = 0.022). LightGBM shows improvement over Ridge but falls short of significance (p = 0.084).
- **MAPE (secondary metric):** No model achieves significant improvement over any baseline. All three tests fail to reject the null hypothesis.

This divergence is not a contradiction — it reflects the fundamental unsuitability of MAPE for sparse demand data.

### 5.2 The MAPE Inflation Problem

MAPE is defined as the mean of |actual - predicted| / |actual|. When actual demand is zero or near-zero — common in our dataset where many product-months have single-digit sales — even small absolute errors produce enormous percentage errors. A prediction of 2 units when actual demand is 1 unit yields 100% MAPE; a prediction of 3 when actual is 1 yields 200%.

This inflation affects all models equally but amplifies variance, making it nearly impossible to detect statistically significant differences. The mean MAPE values (907–1,256%) across models are all inflated by the same set of sparse-demand products, masking genuine improvements in the higher-volume products that matter most for business decisions.

RMSE, by contrast, weights errors by their squared magnitude. Large errors on high-volume products — where forecast accuracy has the greatest business impact — dominate the RMSE calculation, while the numerous small errors on sparse products contribute minimally. This makes RMSE a more meaningful metric for evaluating forecast quality in this context.

### 5.3 Feature Importance Interpretation

The dominance of `quantity_lag_1` (64.4% importance) indicates strong month-to-month autocorrelation in demand. This is expected for consumer goods where recent sales patterns are the strongest predictor of near-term demand. The 3-month lag (14.8%) captures quarterly purchasing cycles, while the COVID-period indicator (2.1%) reflects the structural break that disrupted normal demand patterns.

Hunting season features, while individually contributing less importance, collectively validate the domain-driven feature engineering approach. The turkey spring season and hunting intensity score both appear in the top 15, suggesting that seasonal demand drivers are captured by the model even when lag features are available.

### 5.4 SARIMA Performance

SARIMA's extreme error (RMSE > 283,000) reflects fundamental limitations of univariate parametric models for this dataset. With only 24–45 months of per-product training data and high proportion of convergence failures, SARIMA cannot reliably estimate seasonal parameters for most products. The fallback to training-mean prediction on failed fits further degrades performance. This result supports the decision to focus on feature-engineered supervised models for multi-product demand forecasting.

### 5.5 Limitations

Several limitations should be noted:

1. **Small fold count.** With only 8 cross-validation folds, the paired t-tests have limited statistical power (7 degrees of freedom). Marginal improvements may fail to reach significance due to insufficient sample size rather than absence of effect.

2. **Product aggregation.** Metrics are computed across all products within each fold. Per-product performance varies substantially, and high-volume products disproportionately influence RMSE and MAE.

3. **Temporal scope.** The 2020–2023 period includes the COVID-19 pandemic, which introduced atypical demand patterns. Model performance may differ on more typical market conditions.

4. **Feature leakage risk.** While we excluded rolling statistics that include the current period, the lag-1 feature uses the immediately preceding month, which in practice may not be available at forecast time depending on data latency.

5. **Single domain.** Results are specific to shotgun demand at one distributor. Generalization to other product categories or markets would require additional validation.

## 6. Conclusion

This study evaluated five forecasting approaches for SKU-level shotgun demand prediction using a rigorous rolling-origin cross-validation framework. Our key contributions are:

1. **XGBoost is the recommended forecasting model**, achieving the lowest RMSE (646.2) and MAE (289.6) and significantly outperforming both Ridge and Seasonal Naive baselines on RMSE (p < 0.05).

2. **Metric selection critically affects conclusions.** MAPE-based testing shows no significant differences, while RMSE-based testing reveals significant improvement. For sparse demand data, RMSE should be preferred over MAPE as the primary evaluation metric.

3. **Domain-specific feature engineering adds value.** Hunting season indicators and COVID-period flags contribute to model performance, validating the importance of incorporating domain knowledge into feature design.

4. **SARIMA is unsuitable for multi-product sparse demand.** The univariate parametric approach fails on products with insufficient history and sparse demand patterns.

### Future Work

Several directions could extend this research:

- **Hierarchical forecasting** using reconciliation methods (Hyndman et al., 2011) to leverage category-level patterns for improving SKU-level forecasts.
- **Intermittent demand models** such as Croston's method (Croston, 1972) or SBA (Syntetos & Boylan, 2001) for low-volume products, combined with standard models for high-volume products.
- **External data integration** including NICS background check volumes, economic indicators, and hunting license data as exogenous features.
- **Conformal prediction intervals** (Shafer & Vovk, 2008) to quantify forecast uncertainty for inventory optimization decisions.
- **Extended evaluation period** beyond the COVID-affected window to assess model robustness under more typical market conditions.

## 7. Bibliography

Bergmeir, C., & Benitez, J. M. (2012). On the use of cross-validation for time series predictor evaluation. *Information Sciences*, 191, 192–213.

Bojer, C. S., & Meldgaard, J. P. (2021). Kaggle forecasting competitions: An overlooked learning opportunity. *International Journal of Forecasting*, 37(2), 587–599.

Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785–794.

Croston, J. D. (1972). Forecasting and stock control for intermittent demands. *Journal of the Operational Research Society*, 23(3), 289–303.

Demsar, J. (2006). Statistical comparisons of classifiers over multiple data sets. *Journal of Machine Learning Research*, 7, 1–30.

Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts.

Hyndman, R. J., Ahmed, R. A., Athanasopoulos, G., & Shang, H. L. (2011). Optimal combination forecasts for hierarchical time series. *Computational Statistics & Data Analysis*, 55(9), 2579–2589.

Januschowski, T., Gasthaus, J., Wang, Y., Salinas, D., Flunkert, V., Bohlke-Schneider, M., & Callot, L. (2020). Criteria for classifying forecasting methods. *International Journal of Forecasting*, 36(1), 167–177.

Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. *Advances in Neural Information Processing Systems*, 30, 3146–3154.

Kolassa, S. (2016). Evaluating predictive count data distributions in retail sales forecasting. *International Journal of Forecasting*, 32(3), 788–803.

Lang, M. A. (2013). Firearm commerce in the United States: Annual statistical update. *Bureau of Alcohol, Tobacco, Firearms and Explosives*.

Makridakis, S., Spiliotis, E., & Assimakopoulos, V. (2018). Statistical and machine learning forecasting methods: Concerns and ways forward. *PLoS ONE*, 13(3), e0194889.

National Shooting Sports Foundation (NSSF). (2023). *Firearm and ammunition industry economic impact report*. NSSF.

Shafer, G., & Vovk, V. (2008). A tutorial on conformal prediction. *Journal of Machine Learning Research*, 9, 371–421.

Syntetos, A. A., & Boylan, J. E. (2001). On the bias of intermittent demand estimates. *International Journal of Production Economics*, 71(1–3), 457–466.

Syntetos, A. A., & Boylan, J. E. (2005). The accuracy of intermittent demand estimates. *International Journal of Forecasting*, 21(2), 303–314.

Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: An analysis and review. *International Journal of Forecasting*, 16(4), 437–450.